# Load ChromaDB

In [1]:
# Mount Google Drive to access and save files
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Import chromda db
try:
  import chromadb
except:
  !pip install chromadb
  import chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 102.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 100.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 8.3 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opente

In [3]:
# Set up client and set path to google drive
client = chromadb.PersistentClient(path="/content/drive/MyDrive/chroma_db")

# Initialize movie chunk collection and count entries
collection = client.get_collection(name="movie_chunks")
collection.count()

164714

# Query Embedding Function

In [4]:
# Import sentence transformers
try:
  from sentence_transformers import SentenceTransformer
except:
  !pip install sentence_transformers
  from sentence_transformers import SentenceTransformer

In [5]:
# Load pretrained model
embedding_model = SentenceTransformer("all-mpnet-base-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
def get_query_embedding(query, embedding_model):
  """
  This function takes a input query removes leading and
  lagging whitespaces, then generates an embedding for the query.

  The resulting embedding is converted to a Python list format, making it
  compatible with vector databases such as Chroma.

  Parameters
  ----------
  query: str
    The input query string to be embedded

  Returns
  --------
  embedding_model : object
        A preloaded SentenceTransformer model
        (in this case all-mpnet-base-v2).
  """
  return embedding_model.encode(query.strip()).tolist()

# Retrieval Function

In [7]:
def retrieve_documents(query, collection, embedding_model, top_k=5):
    """
    This function takes an input query, generates its embedding,
    and uses that embedding to retrieve the top-k most relevant
    document chunks from a Chroma collection.

    It returns both the retrieved document texts and their
    corresponding chunk IDs, which can be used for source
    referencing in a RAG prompt.

    Parameters
    ----------
    query : str
        The input query string used for retrieval

    collection : object
        The Chroma collection containing embedded document chunks

    model : object
        A preloaded SentenceTransformer model used to generate the query embedding
    top_k : int, optional
        The number of top matching document chunks to retrieve, the default is 5

    Returns
    -------
    tuple
        A tuple containing:
          documents : list
              A list of the retrieved documents
          ids : list
              A corresponding list of the chunk IDs for the retrieved documents
    """
    # Call query embedding function
    query_embedding = get_query_embedding(query, embedding_model)

    # Query chroma db
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents"]
    )
    # Return docs and ids
    return results["documents"][0], results["ids"][0]

## Test Cases

In [13]:
# Define function for printing outputs from test cases
def print_test_output(documents, ids):
  """
  Doc-string here....
  """
  # View each result
  for i in range(len(documents)):

      # Shorten plots
      preview = documents[i][:300]
      if len(documents[i]) > 300:
          preview += "..."

      # Print result
      print(f"Result {i+1}")
      print(f"ID: {ids[i]}")
      print(f"{preview}")
      print("-" * 80)

### Title-Based

#### 1. What is the plot of The Hunger Games?

In [11]:
query1 = "What is the plot of The Hunger Games?"
documents, ids = retrieve_documents(query1, collection, embedding_model, top_k=3)

In [12]:
# View each result
print_test_output(documents, ids)

Result 1
ID: 31186339_plot_2
Movie Title: The Hunger Games
Wiki ID: 31186339
Realse Date: 2012-03-12
Plot Summary: Katniss has Rue draw them off, then destroys the stockpile by setting off the mines planted around it. Furious, Cato kills the boy assigned to guard it. As Katniss runs from the scene, she hears Rue calling her nam...
--------------------------------------------------------------------------------
Result 2
ID: 31186339_plot_5
Movie Title: The Hunger Games
Wiki ID: 31186339
Realse Date: 2012-03-12
Plot Summary: Haymitch warns Katniss that she has made powerful enemies after her display of defiance. She and Peeta return to District 12, while Crane is locked in a room with a bowl of nightlock berries, and President Snow con...
--------------------------------------------------------------------------------
Result 3
ID: 31186339_plot_0
Movie Title: The Hunger Games
Wiki ID: 31186339
Realse Date: 2012-03-12
Plot Summary: The nation of Panem consists of a wealthy Capitol and twe

#### 2. What happens in the Titanic?

In [ ]:
query2 = "What happens in the Titanic?"
documents, ids = retrieve_documents(query2, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

Result 1
ID: 6524086_plot_0
Movie Title: A Night to Remember
Wiki ID: 6524086
Realse Date: 1958-07-01
Plot Summary: The Titanic was the largest vessel afloat, and was widely believed to be unsinkable. Her passengers included the cream of American and British society. The story of her sinking is told from the point of view of h...
--------------------------------------------------------------------------------
Result 2
ID: 52371_plot_3
Movie Title: Titanic
Wiki ID: 52371
Realse Date: 1997-11-01
Plot Summary: After exhausting his ammunition, Cal realizes to his chagrin that he gave his coat with the diamond to Rose. With the situation now dire, he returns to the boat deck and boards a lifeboat by pretending to look after a lost chi...
--------------------------------------------------------------------------------
Result 3
ID: 6524086_plot_1
Movie Title: A Night to Remember
Wiki ID: 6524086
Realse Date: 1958-07-01
Plot Summary: Fortunately, the radio operator on the receives the distress


#### 3. Give me the story of Star Wars Episode V: The Empire Strikes Back.

In [ ]:
query3 = "Give me the story of Star Wars Episode V: The Empire Strikes Back."
documents, ids = retrieve_documents(query3, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

Result 1
ID: 53964_meta_0
Movie Title: Star Wars Episode V: The Empire Strikes Back
Wiki ID: 53964
Release Date: 1980-05-21
Runtime: 124.0
Languages: English
Countries: United States of America
Genres: Science Fiction, Adventure, Space opera, Fantasy, Family Film, Action
Box Office Revenue: 538375067.0
--------------------------------------------------------------------------------
Result 2
ID: 53964_plot_2
Movie Title: Star Wars Episode V: The Empire Strikes Back
Wiki ID: 53964
Realse Date: 1980-05-21
Plot Summary: Boba Fett secretly follows the Millennium Falcon to the planet Bespin and arrives just before Han and Leia. Bespin is run by Han's old friend Lando Calrissian , but shortly after they arriv...
--------------------------------------------------------------------------------
Result 3
ID: 50744_plot_2
Movie Title: Star Wars Episode VI: Return of the Jedi
Wiki ID: 50744
Realse Date: 1983-05-25
Plot Summary: Palpatine tempts Luke to give in to his anger and join the dark side, a

#### 4. What is the plot of Sex and the City the Movie?

In [ ]:
query4 = "What is the plot of Sex and the City the Movie?"
documents, ids = retrieve_documents(query4, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

Result 1
ID: 13239722_plot_2
Movie Title: Sex and the City: The Movie
Wiki ID: 13239722
Realse Date: 2008-05-12
Plot Summary: Miranda eventually confesses to Carrie about what happened at the rehearsal dinner, and the two briefly fall out. After reflecting on the argument she had with Carrie, Miranda agrees to attend couples co...
--------------------------------------------------------------------------------
Result 2
ID: 13239722_plot_0
Movie Title: Sex and the City: The Movie
Wiki ID: 13239722
Realse Date: 2008-05-12
Plot Summary: Carrie walks through the streets of New York City thinking about events that has happened to her and her friends during Sex and the City. Charlotte is now living a happy marriage with Harry Goldenblatt, ...
--------------------------------------------------------------------------------
Result 3
ID: 23091532_plot_3
Movie Title: Sex and the City 2
Wiki ID: 23091532
Realse Date: 2010-05-27
Plot Summary: Worse, the Sheikh cancels the PR meeting and ceases pay

#### 5. What is the plot of Batman Begins?

In [ ]:
query5 = "What is the plot of Batman Begins?"
documents, ids = retrieve_documents(query5, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

Result 1
ID: 481605_plot_0
Movie Title: Batman Begins
Wiki ID: 481605
Realse Date: 2005-06-10
Plot Summary: As a child, Bruce Wayne falls into a well, developing a fear of bats. Soon afterward, he witnesses his parents' murder by mugger Joe Chill, leaving Bruce to be raised by the family butler, Alfred Pennyworth. Years later...
--------------------------------------------------------------------------------
Result 2
ID: 13375677_plot_2
Movie Title: Batman: Year One
Wiki ID: 13375677
Realse Date: 2011-10-18
Plot Summary: Branden manages to climb out of the trap trough a collapsed chimney, and joins in the gun battle. Enraged as the team’s careless gunfire injures several people outside, Batman throws Branden through a brick wall an...
--------------------------------------------------------------------------------
Result 3
ID: 481605_meta_0
Movie Title: Batman Begins
Wiki ID: 481605
Release Date: 2005-06-10
Runtime: 141.0
Languages: Standard Mandarin, Urdu , English
Countries: United S

### Character-Based

#### 6. What happens between Carrie and Big in Sex and The City the Movie?

In [ ]:
query6 = "What happens between Carrie and Big in Sex and The City the Movie?"
documents, ids = retrieve_documents(query6, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

Result 1
ID: 13239722_plot_0
Movie Title: Sex and the City: The Movie
Wiki ID: 13239722
Realse Date: 2008-05-12
Plot Summary: Carrie walks through the streets of New York City thinking about events that has happened to her and her friends during Sex and the City. Charlotte is now living a happy marriage with Harry Goldenblatt, ...
--------------------------------------------------------------------------------
Result 2
ID: 13239722_plot_2
Movie Title: Sex and the City: The Movie
Wiki ID: 13239722
Realse Date: 2008-05-12
Plot Summary: Miranda eventually confesses to Carrie about what happened at the rehearsal dinner, and the two briefly fall out. After reflecting on the argument she had with Carrie, Miranda agrees to attend couples co...
--------------------------------------------------------------------------------
Result 3
ID: 13239722_plot_3
Movie Title: Sex and the City: The Movie
Wiki ID: 13239722
Realse Date: 2008-05-12
Plot Summary: Carrie searches her correspondence and finds t

#### 7. What happens between Jack and Rose in Titanic?

In [ ]:
query7 = "What happens between Jack and Rose in Titanic? "
documents, ids = retrieve_documents(query7, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

Result 1
ID: 52371_plot_1
Movie Title: Titanic
Wiki ID: 52371
Realse Date: 1997-11-01
Plot Summary: Cal is at first aloof to Jack, but when Rose indicates that recognition is due, he offers him a small amount of money. After Rose mocks Cal for this, asking if her life means so little, he invites Jack to a first-class dinner ...
--------------------------------------------------------------------------------
Result 2
ID: 52371_plot_3
Movie Title: Titanic
Wiki ID: 52371
Realse Date: 1997-11-01
Plot Summary: After exhausting his ammunition, Cal realizes to his chagrin that he gave his coat with the diamond to Rose. With the situation now dire, he returns to the boat deck and boards a lifeboat by pretending to look after a lost chi...
--------------------------------------------------------------------------------
Result 3
ID: 52371_plot_4
Movie Title: Titanic
Wiki ID: 52371
Realse Date: 1997-11-01
Plot Summary: Rose and the other survivors are taken by the RMS Carpathia to New York, where

#### 8. Who is Katniss and what role does she play in The Hunger Games?

In [ ]:
query8 = "Who is Katniss and what role does she play in The Hunger Games?"
documents, ids = retrieve_documents(query8, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

Result 1
ID: 31186339_plot_0
Movie Title: The Hunger Games
Wiki ID: 31186339
Realse Date: 2012-03-12
Plot Summary: The nation of Panem consists of a wealthy Capitol and twelve poorer districts. As punishment for a past rebellion, each district must provide a boy and girl between the ages of 12 and 18 selected by lottery for the...
--------------------------------------------------------------------------------
Result 2
ID: 31186339_cast_0
Movie Title: The Hunger Games
Wiki ID: 31186339
Release Date: 2012-03-12
Characters and Actors:
Foxface:  Jacqueline Emerson
Katniss Everdeen:  Jennifer Lawrence
Peeta Mellark:  Josh Hutcherson
Effie Trinket:  Elizabeth Banks
Gale Hawthorne:  Liam Hemsworth
Haymitch Abernathy:  Woody Harrelson
Clove...
--------------------------------------------------------------------------------
Result 3
ID: 31186339_plot_1
Movie Title: The Hunger Games
Wiki ID: 31186339
Realse Date: 2012-03-12
Plot Summary: However, she discovers Peeta meant what he said. The tele

#### 9. What is the relationship between Luke and Darth Vader in Star Wars Episode V?

In [ ]:
query9 = "What is the relationship between Luke and Darth Vader in Star Wars Episode V?"
documents, ids = retrieve_documents(query9, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

Result 1
ID: 50744_plot_0
Movie Title: Star Wars Episode VI: Return of the Jedi
Wiki ID: 50744
Realse Date: 1983-05-25
Plot Summary: Luke Skywalker initiates a plan to rescue Han Solo from the crime lord Jabba the Hutt with the help of Princess Leia, Lando Calrissian, Chewbacca, C-3PO and R2-D2. Leia infiltrates Jabba's pala...
--------------------------------------------------------------------------------
Result 2
ID: 53964_plot_1
Movie Title: Star Wars Episode V: The Empire Strikes Back
Wiki ID: 53964
Realse Date: 1980-05-21
Plot Summary: Luke escapes with R2-D2 in an X-wing fighter and crash lands on Dagobah. He is soon found by the diminutive Yoda, who at first pretends to be a simple swamp inhabitant in order to test Luke...
--------------------------------------------------------------------------------
Result 3
ID: 52549_plot_0
Movie Title: Star Wars Episode IV: A New Hope
Wiki ID: 52549
Realse Date: 1977-05-25
Plot Summary: The film begins with an opening crawl explaining that

#### 10. Tell me about Darth Vader

In [ ]:
query10 = "Tell me about Darth Vader"
documents, ids = retrieve_documents(query10, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

Result 1
ID: 55447_plot_2
Movie Title: Star Wars Episode III: Revenge of the Sith
Wiki ID: 55447
Realse Date: 2005-05-15
Plot Summary: Vader and Obi-Wan engage in a fierce duel, taking them across the entire volcano, and eventually Obi-Wan severs Vader's legs and left arm and Vader falls down to the edge of the volcanic rive...
--------------------------------------------------------------------------------
Result 2
ID: 55447_plot_1
Movie Title: Star Wars Episode III: Revenge of the Sith
Wiki ID: 55447
Realse Date: 2005-05-15
Plot Summary: When Palpatine reveals himself as the Sith lord Darth Sidious, Anakin reports his treachery to Jedi Master Mace Windu. In the ensuing lightsaber duel, Windu subdues Palpatine by deflecting hi...
--------------------------------------------------------------------------------
Result 3
ID: 52549_plot_0
Movie Title: Star Wars Episode IV: A New Hope
Wiki ID: 52549
Realse Date: 1977-05-25
Plot Summary: The film begins with an opening crawl explaining that

### Metadata Based

#### 11. When was the Titanic released?

In [ ]:
query = "When was the Titanic released?"
documents, ids = retrieve_documents(query, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

Result 1
ID: 1203778_meta_0
Movie Title: Titanic
Wiki ID: 1203778
Release Date: 1953-04-16
Runtime: 103.0
Languages: English
Countries: United States of America
Genres: Black-and-white, Action/Adventure, Drama, Disaster, Romantic drama, Romance Film, Action
Box Office Revenue: 2250000.0
--------------------------------------------------------------------------------
Result 2
ID: 2052143_meta_0
Movie Title: Titanic
Wiki ID: 2052143
Release Date: 1943-11-10
Runtime: 85.0
Languages: German
Countries: Germany
Genres: Disaster, Propaganda film, Drama, World cinema, Black-and-white
Box Office Revenue: nan
--------------------------------------------------------------------------------
Result 3
ID: 52371_meta_0
Movie Title: Titanic
Wiki ID: 52371
Release Date: 1997-11-01
Runtime: 194.0
Languages: Italian , English , French , Swedish , Russian , German
Countries: United States of America
Genres: Tragedy, Costume drama, Historical fiction, Action/Adventure, Period piece, Drama, Disaster, Romant

#### 12. What genre is The Hunger Games?

In [ ]:
query = "What genre is The Hunger Games?"
documents, ids = retrieve_documents(query, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

Result 1
ID: 31186339_meta_0
Movie Title: The Hunger Games
Wiki ID: 31186339
Release Date: 2012-03-12
Runtime: 142.0
Languages: English
Countries: United States of America
Genres: Action/Adventure, Science Fiction, Action, Drama
Box Office Revenue: 686533290.0
--------------------------------------------------------------------------------
Result 2
ID: 31186339_plot_0
Movie Title: The Hunger Games
Wiki ID: 31186339
Realse Date: 2012-03-12
Plot Summary: The nation of Panem consists of a wealthy Capitol and twelve poorer districts. As punishment for a past rebellion, each district must provide a boy and girl between the ages of 12 and 18 selected by lottery for the...
--------------------------------------------------------------------------------
Result 3
ID: 36012788_plot_0
Movie Title: The Hunger Games: Catching Fire
Wiki ID: 36012788
Realse Date: Unknown
Plot Summary: The Hunger Games: Catching Fire begins as Katniss Everdeen has returned home safe after winning the 74th Annual Hunge

#### 13. What language is Spirited Away in?

In [ ]:
query = "What language is Spirited Away in?"
documents, ids = retrieve_documents(query, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

Result 1
ID: 70091_meta_0
Movie Title: Spirited Away
Wiki ID: 70091
Release Date: 2001-07-20
Runtime: 123.0
Languages: Japanese
Countries: Japan
Genres: Japanese Movies, Adventure, World cinema, Animation, Fantasy, Family Film
Box Office Revenue: nan
--------------------------------------------------------------------------------
Result 2
ID: 70091_cast_0
Movie Title: Spirited Away
Wiki ID: 70091
Release Date: 2001-07-20
Characters and Actors:
Bô:  Ryunosuke Kamiki
Haku:  Jason Marsden,  Miyu Irino
Yuuko Ogino:  Lauren Holly,  Yasuko Sawaguchi
Akio Ogino:  Michael Chiklis,  Takashi Naitō
Chihiro Ogino:  Daveigh Chase,  Rumi Hiiragi
Yubaba:  Nina Hage...
--------------------------------------------------------------------------------
Result 3
ID: 70091_plot_0
Movie Title: Spirited Away
Wiki ID: 70091
Realse Date: 2001-07-20
Plot Summary: Ten-year-old Chihiro Ogino and her parents are traveling to their new home when her father takes a wrong turn. Thinking that they have found an abandon

### Conceptual-Based

#### 14. Which movie is about a sinking ship and a romance?

In [ ]:
query = "Which movie is about a sinking ship and a romance?"
documents, ids = retrieve_documents(query, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

Result 1
ID: 11841923_plot_2
Movie Title: Sound of the Sea
Wiki ID: 11841923
Realse Date: 2001-06-08
Plot Summary: He sabotages the Son de Mar, which they use to escape. In the middle of the ocean, the ship sinks and the lovers find peace and eternal love in death.
--------------------------------------------------------------------------------
Result 2
ID: 14876301_meta_0
Movie Title: Romance on the High Seas
Wiki ID: 14876301
Release Date: 1948-07-03
Runtime: 99.0
Languages: English
Countries: United States of America
Genres: Romantic comedy, Romance Film, Musical, Comedy, Musical comedy
Box Office Revenue: nan
--------------------------------------------------------------------------------
Result 3
ID: 8522084_plot_0
Movie Title: Atlantic
Wiki ID: 8522084
Realse Date: 1930-10-13
Plot Summary: Atlantic is a drama film based on the RMS Titanic and set aboard a fictional ship, called the Atlantic. The White Star Line nevertheless did at one time own a major liner called Atlantic which 

#### 15. Which movie includes a rebellion against the Capitol?

In [ ]:
query = "Which movie includes a rebellion against the Capitol?"
documents, ids = retrieve_documents(query, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

Result 1
ID: 24881386_plot_0
Movie Title: The Fall of a Nation
Wiki ID: 24881386
Realse Date: 1916-06-06
Plot Summary: The Fall of a Nation is an attack on the pacifism of William Jennings Bryan and Henry Ford<ref nameRothstein first titleMovies.nytimes.com date2009-10-30 and a plea for American preparedness for war. America is...
--------------------------------------------------------------------------------
Result 2
ID: 24840023_meta_0
Movie Title: Rebellion
Wiki ID: 24840023
Release Date: 2009
Runtime: 100.0
Languages: Standard Mandarin, Cantonese
Countries: Hong Kong
Genres: Thriller, Action/Adventure, Chinese Movies, World cinema, Crime Thriller
Box Office Revenue: nan
--------------------------------------------------------------------------------
Result 3
ID: 29565990_plot_1
Movie Title: Revolt of the Praetorians
Wiki ID: 29565990
Realse Date: 1964
Plot Summary: He tells of Domitian and Artamne's plan to arrest and execute all patricians suspected of treason, and Valerius, as t

#### 16. Which movie features the Battle of Hoth?

In [ ]:
query = "Which movie features the Battle of Hoth?"
documents, ids = retrieve_documents(query, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

Result 1
ID: 4876314_cast_0
Movie Title: The Battle of Kerzhenets
Wiki ID: 4876314
Release Date: 1971
Characters and Actors:
Unknown
--------------------------------------------------------------------------------
Result 2
ID: 4240215_meta_0
Movie Title: The Battle of Shaker Heights
Wiki ID: 4240215
Release Date: 2003-08-22
Runtime: 90.0
Languages: English , German
Countries: United States of America
Genres: Indie, Comedy-drama, Coming of age, Comedy, Drama, Romantic drama, Romance Film, Teen
Box Office Revenue: 280351.0
--------------------------------------------------------------------------------
Result 3
ID: 12265808_meta_0
Movie Title: Juken Sentai Gekiranger: Nei-Nei! Hō-Hō! Hong Kong Decisive Battle
Wiki ID: 12265808
Release Date: 2007-08-04
Runtime: 33.0
Languages: Japanese
Countries: Japan
Genres: Short Film, Japanese Movies
Box Office Revenue: nan
--------------------------------------------------------------------------------


### Ambiguous-Based

#### 17. tell me about the movie where the ship sinks and the couple falls in love

In [ ]:
query = "tell me about the movie where the ship sinks and the couple falls in love"
documents, ids = retrieve_documents(query, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

Result 1
ID: 11841923_plot_2
Movie Title: Sound of the Sea
Wiki ID: 11841923
Realse Date: 2001-06-08
Plot Summary: He sabotages the Son de Mar, which they use to escape. In the middle of the ocean, the ship sinks and the lovers find peace and eternal love in death.
--------------------------------------------------------------------------------
Result 2
ID: 28858014_plot_2
Movie Title: Submarine
Wiki ID: 28858014
Realse Date: 2010-09-12
Plot Summary: Together they walk into the sea smiling. It is hinted that the two resume their relationship.
--------------------------------------------------------------------------------
Result 3
ID: 8522084_plot_0
Movie Title: Atlantic
Wiki ID: 8522084
Realse Date: 1930-10-13
Plot Summary: Atlantic is a drama film based on the RMS Titanic and set aboard a fictional ship, called the Atlantic. The White Star Line nevertheless did at one time own a major liner called Atlantic which was lost in 1873. The main plo...
--------------------------------------

#### 18. the hunger games movie plot

In [ ]:
query = "the hunger games movie plot"
documents, ids = retrieve_documents(query, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

Result 1
ID: 31186339_plot_2
Movie Title: The Hunger Games
Wiki ID: 31186339
Realse Date: 2012-03-12
Plot Summary: Katniss has Rue draw them off, then destroys the stockpile by setting off the mines planted around it. Furious, Cato kills the boy assigned to guard it. As Katniss runs from the scene, she hears Rue calling her nam...
--------------------------------------------------------------------------------
Result 2
ID: 36012788_plot_0
Movie Title: The Hunger Games: Catching Fire
Wiki ID: 36012788
Realse Date: Unknown
Plot Summary: The Hunger Games: Catching Fire begins as Katniss Everdeen has returned home safe after winning the 74th Annual Hunger Games along with fellow tribute Peeta Mellark . Winning means that they must turn a...
--------------------------------------------------------------------------------
Result 3
ID: 31186339_plot_5
Movie Title: The Hunger Games
Wiki ID: 31186339
Realse Date: 2012-03-12
Plot Summary: Haymitch warns Katniss that she has made powerful enemies

#### 20. when did that star wars empire movie come out

In [ ]:
query = "when did that star wars empire movie come out"
documents, ids = retrieve_documents(query, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

Result 1
ID: 1958896_meta_0
Movie Title: Empire
Wiki ID: 1958896
Release Date: 2002-12-03
Runtime: 90.0
Languages: English
Countries: United States of America
Genres: Thriller, Crime Fiction, Action, Drama, Indie
Box Office Revenue: 18591272.0
--------------------------------------------------------------------------------
Result 2
ID: 1236676_cast_0
Movie Title: Empire
Wiki ID: 1236676
Release Date: 1964
Characters and Actors:
Unknown
--------------------------------------------------------------------------------
Result 3
ID: 34272720_cast_0
Movie Title: Evil Empire: A Talk by Chalmers Johnson
Wiki ID: 34272720
Release Date: 2007
Characters and Actors:
Unknown Character:  Chalmers Johnson
--------------------------------------------------------------------------------


### No Retrival Expected

In [ ]:
query = "What is the plot of Avatar 2?"
documents, ids = retrieve_documents(query, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

Result 1
ID: 4273140_meta_0
Movie Title: Avatar
Wiki ID: 4273140
Release Date: 2009-12-10
Runtime: 178.0
Languages: English , Spanish
Countries: United States of America, United Kingdom
Genres: Thriller, Science Fiction, Adventure, Computer Animation, Epic, Nature, Fantasy, Action
Box Office Revenue: 2782275172.0
--------------------------------------------------------------------------------
Result 2
ID: 15945267_meta_0
Movie Title: Avatar
Wiki ID: 15945267
Release Date: 2004
Runtime: 90.0
Languages: English
Countries: Singapore
Genres: Thriller, Science Fiction, Adventure
Box Office Revenue: nan
--------------------------------------------------------------------------------
Result 3
ID: 4273140_cast_0
Movie Title: Avatar
Wiki ID: 4273140
Release Date: 2009-12-10
Characters and Actors:
Eytukan:  Wes Studi
Jake Sully:  Sam Worthington
Neytiri:  Zoe Saldana
Dr. Grace Augustine:  Sigourney Weaver
Colonel Miles Quaritch:  Stephen Lang
Trudy Chacon:  Michelle Rodriguez
Parker Selfridge:  

In [ ]:
query = "What happens in Inception?"
documents, ids = retrieve_documents(query, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

Result 1
ID: 23270459_plot_2
Movie Title: Inception
Wiki ID: 23270459
Realse Date: 2010-07-08
Plot Summary: Unbeknownst to the rest of the team, due to the effects of heavy sedation and multi-layered dreaming, death during this mission will result in entering Limbo, unconstructed dream space where the dreamer could be trapped i...
--------------------------------------------------------------------------------
Result 2
ID: 23270459_meta_0
Movie Title: Inception
Wiki ID: 23270459
Release Date: 2010-07-08
Runtime: 148.0
Languages: French , Japanese , English
Countries: United States of America, United Kingdom
Genres: Thriller, Science Fiction, Adventure, Psychological thriller, Mystery, Action
Box Office Revenue: 825532764.0
--------------------------------------------------------------------------------
Result 3
ID: 23270459_cast_0
Movie Title: Inception
Wiki ID: 23270459
Release Date: 2010-07-08
Characters and Actors:
Yusuf:  Dileep Rao
Cobb:  Leonardo DiCaprio
Arthur:  Joseph Gordon-L

In [ ]:
query = "Tell me about the WWE"
documents, ids = retrieve_documents(query, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

Result 1
ID: 15122117_cast_0
Movie Title: The Wrestler
Wiki ID: 15122117
Release Date: 2008-09-05
Characters and Actors:
Necro Butcher:  Necro Butcher
Randy Robinson:  Mickey Rourke
Cassidy:  Marisa Tomei
Stephanie:  Evan Rachel Wood
Lenny:  Mark Margolis
Wayne:  Todd Barry
Nick Volpe:  Wass M. Stevens
Scott Brumberg:  Judah Fr...
--------------------------------------------------------------------------------
Result 2
ID: 29894989_plot_0
Movie Title: NWF Kids Pro Wrestling: The Untold Story
Wiki ID: 29894989
Realse Date: 2005
Plot Summary: The documentary covers the birth and development of a professional style wrestling league from the mid 1980s that was produced by young teens aging from 12 to 16 years of age. The film covers the ...
--------------------------------------------------------------------------------
Result 3
ID: 10301701_meta_0
Movie Title: The Wrestlers
Wiki ID: 10301701
Release Date: 2000
Runtime: 99.0
Languages: Bengali
Countries: India
Genres: Thriller, Drama
Box O

In [ ]:
query = "What is the plot of the banana spaceship movie?"
documents, ids = retrieve_documents(query, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

Result 1
ID: 1329111_meta_0
Movie Title: Bananas
Wiki ID: 1329111
Release Date: 1971-09
Runtime: 82.0
Languages: English
Countries: United States of America
Genres: Parody, Absurdism, Political satire, Satire, Comedy, Political cinema, Slapstick
Box Office Revenue: 11833696.0
--------------------------------------------------------------------------------
Result 2
ID: 5172645_meta_0
Movie Title: Top Banana
Wiki ID: 5172645
Release Date: 1954-01-27
Runtime: 100.0
Languages: English
Countries: United States of America
Genres: Musical, Comedy, Musical comedy
Box Office Revenue: nan
--------------------------------------------------------------------------------
Result 3
ID: 33777905_meta_0
Movie Title: Pink Bananas
Wiki ID: 33777905
Release Date: 1978-12-22
Runtime: nan
Languages: nan
Countries: nan
Genres: Short Film, Comedy film, Animation
Box Office Revenue: nan
--------------------------------------------------------------------------------


In [ ]:
query = "Who is the main character in the invisible dragon film?"
documents, ids = retrieve_documents(query, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

Result 1
ID: 22644547_cast_0
Movie Title: The Dragon That Wasn't
Wiki ID: 22644547
Release Date: 1983-02-03
Characters and Actors:
Unknown
--------------------------------------------------------------------------------
Result 2
ID: 32307438_cast_0
Movie Title: Dragon
Wiki ID: 32307438
Release Date: 2006-12-12
Characters and Actors:
Unknown Character:  Eliza Swenson,  Matthew Wolf,  Amelia Jackson-Gray,  Jon-Paul Gates,  Rachel Haines,  Jessica Bork,  Jeff Denton,  Matt Wolf
--------------------------------------------------------------------------------
Result 3
ID: 34993670_cast_0
Movie Title: Lucky Dragon No. 5
Wiki ID: 34993670
Release Date: 1959-02-18
Characters and Actors:
Unknown
--------------------------------------------------------------------------------


# Answer Generating

In [8]:
!pip install -q transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.7 MB/s eta 0:00:00


In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Define the 4-bit configuration
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

model_id = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# Loading model in 4-bit to fit Mistral-7B in T4 memory
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [23]:
def generate_answer(query, context, context_ids, model, tokenizer):
    """
    ConcatenAtes relevant movie chunks and asks Mistral to answer the query based on them.
    """
    # Combine the retrieved docs into a single text block
    context_block = "\n\n".join(context)

    # Create the prompt using Mistral's specific instruction format
    prompt = f"""<s>[INST] You are an expert movie assistant. Use the following context to answer the question.
    If you don't know the answer based on the context, just say you don't know. Do not spoil any movies that you mention.

    Context:
    {context}

    Question: {query}
    [/INST]"""

    # Convert prompt to tokens and send to GPU
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # Generate the answer
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7,          # Temperature is set to 0.7, seems to be decent
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )


    # Calculate the exact length of the input prompt
    input_length = inputs['input_ids'].size(1)

    # Slice the output tensor to ignore the prompt, then decode only the new tokens
    answer_only = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

    return answer_only.strip()

# Full RAG Pipeline (Testing)

In [20]:
def rag_pipeline(query, collection, embedding_model, model, tokenizer, top_k = 3):
  """
  Retrieve the relevant movies based on the query and send that to the model for an answer
  """
  # Retrieve the context documents
  context, context_ids = retrieve_documents(query, collection, embedding_model, top_k = top_k)

  # Print the retrieved documents
  print_test_output(context, context_ids)

  # Generate the answer with the retrieved context
  answer = generate_answer(query, context, context_ids, model, tokenizer)

  return answer

In [18]:
import textwrap

In [21]:
query = "What is a good movie about a road trip gone wrong?"
answer = rag_pipeline(query, collection, embedding_model, model, tokenizer, top_k = 5)
print("\nMistral Response:\n")
print(textwrap.fill(answer, width=80))

Result 1
ID: 1164535_plot_3
Movie Title: Road Trip
Wiki ID: 1164535
Realse Date: 2000-05-11
Plot Summary: Beth then goes to the girls' showers where everyone is naked for some advice. They suggest that she should go to Boston to tell Tiffany about the situation, which Beth does. However, the Tiffany she confronts in Boston is ...
--------------------------------------------------------------------------------
Result 2
ID: 1164535_meta_0
Movie Title: Road Trip
Wiki ID: 1164535
Release Date: 2000-05-11
Runtime: 94.0
Languages: English
Countries: United States of America
Genres: Sex comedy, Stoner film, Adventure, Comedy
Box Office Revenue: 119754278.0
--------------------------------------------------------------------------------
Result 3
ID: 10133100_meta_0
Movie Title: College Road Trip
Wiki ID: 10133100
Release Date: 2008-03-07
Runtime: 83.0
Languages: English
Countries: United States of America
Genres: Road movie, Children's, Coming of age, Comedy, Drama, Family Film, Domestic Comed

In [22]:
query = "I'm craving a good movie about being put into a new environment and having to adjust and eventually thrive?"
answer = rag_pipeline(query, collection, embedding_model, model, tokenizer, top_k = 5)
print("\nMistral Response:\n")
print(textwrap.fill(answer, width=80))

Result 1
ID: 30042062_plot_0
Movie Title: The Newcomers
Wiki ID: 30042062
Realse Date: 2000
Plot Summary: A Boston family decides to move to the country and begin a new life after their young son, Sam, is threatened by bullies. When the Docherty family reaches their new home, things aren't much better. The father, Gary, learns ...
--------------------------------------------------------------------------------
Result 2
ID: 17090332_cast_0
Movie Title: Stories of Change
Wiki ID: 17090332
Release Date: 2008
Characters and Actors:
Unknown
--------------------------------------------------------------------------------
Result 3
ID: 31804235_plot_0
Movie Title: Return
Wiki ID: 31804235
Realse Date: 2011-05-14
Plot Summary: Kelli, returning from her military tour of duty in the Middle East, has high hopes for resuming her old life in her midwestern hometown. Her hopes are gradually dashed as her relationships with her family and friends suffer;...
--------------------------------------------

In [24]:
query = "What are some good movies to put on the TV during a family gathering?"
answer = rag_pipeline(query, collection, embedding_model, model, tokenizer, top_k = 5)
print("\nMistral Response:\n")
print(textwrap.fill(answer, width=80))

Result 1
ID: 16285439_meta_0
Movie Title: Family Viewing
Wiki ID: 16285439
Release Date: 1987-10-02
Runtime: 86.0
Languages: nan
Countries: Canada
Genres: Romance Film, Drama, Indie
Box Office Revenue: nan
--------------------------------------------------------------------------------
Result 2
ID: 20351304_plot_0
Movie Title: The Gathering
Wiki ID: 20351304
Realse Date: 1977-12-04
Plot Summary: Adam Thornton , an ill-tempered executive who walked out on his family, learns that he only has a little time left to live. He decides that he wants to make peace with his family and have one last reunion. He confides...
--------------------------------------------------------------------------------
Result 3
ID: 1373876_meta_0
Movie Title: My Family
Wiki ID: 1373876
Release Date: 1995-05-03
Runtime: 126.0
Languages: English , Spanish
Countries: United States of America
Genres: Period piece, Drama, Indie
Box Office Revenue: nan
-------------------------------------------------------------------